### **Extraer el `nombre del libro`, `autor` y `duracion` de todos los libros que se encuentran en `todas las páginas`**

Vamos a scrapear el siguiente sitio:

https://www.audible.com/search

Vamos a inspeccionar la barra de paginación:  **`< 1 2 ... 25 >`**

<center><img src="https://i.postimg.cc/D0347d0w/ws-307.png"></center>

El tag `<ul>` nos indica toda la barra de paginación:

<center><img src="https://i.postimg.cc/7h1GhGJM/ws-308.png"></center>

Si scroleamos en el tag `<ul>` nos encontramos con los tag `<li>` que nos indican cada una de las páginas:

<center><img src="https://i.postimg.cc/jdXnrfbR/ws-309.png"></center>

Para obtener la última página, utilizamos los indices en reversa:

<center><img src="https://i.postimg.cc/h4V7pwZz/ws-310.png"></center>

Para obtener el botón para avanzar la página, utilizamos el siguiente tag:

<center><img src="https://i.postimg.cc/yYB35P2f/ws-311.png"></center>

Luego, vamos a inspeccionar el sitio y ubicaremos primero la caja principal que contenga todos los titulos:

<center><img src="https://i.postimg.cc/7hLLXGJV/ws-301.png"></center>
<center><img src="https://i.postimg.cc/Kc9jjNCs/ws-302.png"></center>

Buscamos la caja que contenga `la informacion de cada titulo`:

<center><img src="https://i.postimg.cc/nrwhWFvt/ws-303.png"></center>

Buscamos el tag que contenga el `nombre` de cada titulo:

<center><img src="https://i.postimg.cc/HsVxyMgW/ws-304.png"></center>

Buscamos el tag que contenga el `autor` de cada titulo:

<center><img src="https://i.postimg.cc/pX4dYdWf/ws-305.png"></center>

Buscamos el tag que contenga la `duración` de cada titulo:

<center><img src="https://i.postimg.cc/7ZzZByhp/ws-306.png"></center>

Por tanto escribimos el siguiente código:

In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
import pandas as pd
import time

# Modo Headless
options = Options()  # Inicializar una instancia de la clase Options
options.headless = False  # False -> Modo Headless desactivado
# options.add_argument('window-size=1920x1080')  # Establece un tamaño de ventana grande, para que se muestren todos los datos

website = "https://www.audible.com/search"
# Se recomienda utilizar Chrome, pero podriamos utilizar Firefox, Safari, Edge, etc.
driver = webdriver.Chrome()
driver.get(website)
driver.maximize_window()

# Paginación 1
# Localización de la barra de paginación
paginacion = driver.find_element(by=By.XPATH, value='//ul[contains(@class, "pagingElements")]')  
# Localizar cada página mostrada en la barra de paginación
paginas = paginacion.find_elements(by=By.TAG_NAME, value='li') 
# Obtener la última página con indexación negativa (comienza desde donde termina el array)
ultima_pagina = int(paginas[-2].text) 

# Inicializando el almacenamiento
titulo_libro = []
autor_libro = []
duracion_libro = []

# Paginación 2
# Esta es la página que el bot comienza a scrapear
pagina_actual = 1   

# El bucle while funcionará hasta que el bot llegue a la última página del sitio web, entonces se interrumpirá (break)
while pagina_actual <= ultima_pagina:
    time.sleep(2)  # Dejar que la página se muestre correctamente
    # Localizar la caja que contiene todos los audiolibros listados en la página
    container = driver.find_element(by=By.CLASS_NAME, value='adbl-impression-container ')
    # Obtener todos los audiolibros listados (el "/" da nodos hijos inmediatos)
    libros = container.find_elements(by=By.XPATH, value='.//li[contains(@class, "productListItem")]')

    # Recorrer la lista de libros (cada "libro" es un audiolibro)
    for libro in libros:
        # Usamos "contains" para buscar elementos web que contengan un texto concreto, así evitamos construir XPATH largos
        titulo_libro.append(libro.find_element(by=By.XPATH, value='.//h3[contains(@class, "bc-heading")]').text)  # Almacenar los datos en una lista
        autor_libro.append(libro.find_element(by=By.XPATH, value='.//li[contains(@class, "authorLabel")]').text)
        duracion_libro.append(libro.find_element(by=By.XPATH, value='.//li[contains(@class, "runtimeLabel")]').text)

    # Incrementar la página_actual en 1 después de extraer los datos
    pagina_actual = pagina_actual + 1 
    # Localizar el botón 'siguiente_pagina' y pulsar sobre él. Si el elemento no está en la página web, pasar a la siguiente iteración.
    try:
        siguiente_pagina = driver.find_element(by=By.XPATH, value='.//span[contains(@class , "nextButton")]')
        siguiente_pagina.click()
    except:
        pass

driver.quit()

# Almacenar los datos en un DataFrame y exportar a un archivo CSV
df_books = pd.DataFrame({'titulo': titulo_libro, 'autor': autor_libro, 'duracion': duracion_libro})
df_books.to_csv('libros_paginacion.csv', index=False)

Otra forma de trabajar es utilizando **`waits implicitos`** y **`waits explicitos`**:

Un "`wait implícito`" se utiliza para indicar al driver que espere un cierto tiempo cuando intenta localizar un elemento. En Python, puedes importar la librería "`time`" y luego hacer una espera implícita con "`time.sleep()`", y dentro de paréntesis especificas los segundos que quieres que espere el driver. Por ejemplo, si quieres una espera de dos segundos, escribes "`time.sleep(2)`"

Por otro lado, un "`wait explícito`" hace que el web driver espere a que se produzca una condición específica antes de continuar con la ejecución. Para trabajar con la "espera explícita" tenemos que importar tres librerías: `By`, `WebDriverWait` y `EC`, que significa condiciones esperadas (**Expected Conditions**).

In [2]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import time

# Modo Headless
options = Options()  # Inicializar una instancia de la clase Options
options.headless = False  # False -> Modo Headless desactivado
# options.add_argument('window-size=1920x1080')  # Establece un tamaño de ventana grande, para que se muestren todos los datos

website = "https://www.audible.com/search"
# Se recomienda utilizar Chrome, pero podriamos utilizar Firefox, Safari, Edge, etc.
driver = webdriver.Chrome()
driver.get(website)
driver.maximize_window()

# Paginación 1
# Localización de la barra de paginación
paginacion = driver.find_element(by=By.XPATH, value='//ul[contains(@class, "pagingElements")]')  
# Localizar cada página mostrada en la barra de paginación
paginas = paginacion.find_elements(by=By.TAG_NAME, value='li') 
# Obtener la última página con indexación negativa (comienza desde donde termina el array)
ultima_pagina = int(paginas[-2].text) 

# Inicializando el almacenamiento
titulo_libro = []
autor_libro = []
duracion_libro = []

# Paginación 2
# Esta es la página que el bot comienza a scrapear
pagina_actual = 1   

# El bucle while funcionará hasta que el bot llegue a la última página del sitio web, entonces se interrumpirá (break)
while pagina_actual <= ultima_pagina:
    # Wait implicito
    time.sleep(2)  # Dejar que la página se muestre correctamente

    # Wait explicito
    # Localizar la caja que contiene todos los audiolibros listados en la página
    # container = driver.find_element(by=By.CLASS_NAME, value='adbl-impression-container ')
    container = WebDriverWait(driver, 5).until(EC.presence_of_element_located((By.CLASS_NAME, 'adbl-impression-container ')))
    # Obtener todos los audiolibros listados (el "/" da nodos hijos inmediatos)
    # libros = container.find_elements(by=By.XPATH, value='.//li[contains(@class, "productListItem")]')
    libros = WebDriverWait(container, 5).until(EC.presence_of_all_elements_located((By.XPATH, './/li[contains(@class, "productListItem")]')))

    # Recorrer la lista de libros (cada "libro" es un audiolibro)
    for libro in libros:
        # Usamos "contains" para buscar elementos web que contengan un texto concreto, así evitamos construir XPATH largos
        titulo_libro.append(libro.find_element(by=By.XPATH, value='.//h3[contains(@class, "bc-heading")]').text)  # Almacenar los datos en una lista
        autor_libro.append(libro.find_element(by=By.XPATH, value='.//li[contains(@class, "authorLabel")]').text)
        duracion_libro.append(libro.find_element(by=By.XPATH, value='.//li[contains(@class, "runtimeLabel")]').text)

    # Incrementar la página_actual en 1 después de extraer los datos
    pagina_actual = pagina_actual + 1 
    # Localizar el botón 'siguiente_pagina' y pulsar sobre él. Si el elemento no está en la página web, pasar a la siguiente iteración.
    try:
        siguiente_pagina = driver.find_element(by=By.XPATH, value='.//span[contains(@class , "nextButton")]')
        siguiente_pagina.click()
    except:
        pass

driver.quit()

# Almacenar los datos en un DataFrame y exportar a un archivo CSV
df_books = pd.DataFrame({'titulo': titulo_libro, 'autor': autor_libro, 'duracion': duracion_libro})
df_books.to_csv('libros_paginacion_waits.csv', index=False)